# Study Assistant using RAG Llama3.1:8b

In this notebook, we build a RAG pipeline that function as a study assistant. The assistant is fed with personal notes and a textbook, and answers questions grounded in that material to make finding information easier. The project is build with Llama 3.1:8b as the generator that will be delivered **without LangChain** to easily navigate the details of each component added to the pipeline.

Here is the pipeline overview to easily navigate the project:

**Ingestion**

PDF Files → Extract Text (`PyMuPDF`) → Clean Text → Chunk Text → Embed Chunks → Store in ChromaDB

**Query**

User Query → Embed Query → Retrieve Top-k Chunks → Build Prompt → Generate Answer (`OllamaLM`) → Return Answer + Sources

Below we begin with importing necessary libraries that's being used throughout the project.

In [ ]:
# Run this cell once to install dependencies if needed
# !pip install pymupdf sentence-transformers chromadb ollama --break-system-packages -q

In [ ]:
# Import necessary libraries

import pymupdf 
import re
import uuid
import textwrap
from sentence_transformers import SentenceTransformer
import chromadb
import ollama
import pandas as pd

# 1. Data ingestion

The process begins with importing the dataset by defining the file paths. The PDFs will be extracted in **blocks**, which means PyMuPDF splits each page into separate paragraph-like sections based on layout, rather than relying on line breaks. We then apply regex processing to remove hyphens, join wrapped lines, and recombine the blocks back into one single text entry per page, with paragraph breaks preserved between them.

In [ ]:
# Add file paths
PDF_PATHS = [
    "./data_source/personal_notes.pdf",
    "./data_source/textbook.pdf",
]

def clean_page_text(text_or_blocks, strip_tables=True): 
    """Cleans the text extracted from a PDF page."""

    if isinstance(text_or_blocks, list):
        block_texts = []
        for block in text_or_blocks:
            block_text = block[4]  # block[4] is the text content
            block_text = re.sub(r'(\w)-\n(\w)', r'\1\2', block_text)  # fix hyphen breaks
            block_text = re.sub(r'\n', ' ', block_text).strip()       # join lines within block
            if block_text:
                block_texts.append(block_text)
        text = '\n\n'.join(block_texts) # join blocks while preserving paragraph breaks
    else:
        text = text_or_blocks

    text = re.sub(r'[ \t]{2,}', ' ', text).strip()
    return text

def extract_pdf_pages(path, strip_tables=True):
    """Extracts text from each page of a PDF file and returns a list of dictionaries containing the cleaned text, source filename, and page number."""
    
    doc = pymupdf.open(path)
    pages = []

    for page_num, page in enumerate(doc, start=1):
        raw_blocks = page.get_text("blocks")  # get raw text blocks from the page
        cleaned = clean_page_text(raw_blocks, strip_tables=strip_tables)

        if cleaned:
            pages.append({
                "text": cleaned,
                "source": path.split("/")[-1],
                "page": page_num,
            })
    doc.close()

    return pages

all_pages = []
for p in PDF_PATHS:
    all_pages.extend(extract_pdf_pages(p))
print(f"Extracted {len(all_pages)} pages from {len(PDF_PATHS)} file(s).")

# Preview
print("\n\nPreview of the extracted page")
print("="*50)
print(all_pages[0]["text"][:500])


Extracted 72 pages from 2 file(s).


Preview of the extracted page
Text representation

A. Bag-of-words (BOW) and One-hot encoding

Bag-of-Words (BoW): A text representation technique where a document is represented as a vector of word counts or occurrences in a vocabulary. •

One-Hot Encoding: A technique that converts categorical variables (including words) into binary vectors, where only one position in the vector is "hot" (1), and the rest are "cold" (0). •

Example D1 - “I am very happy today” D2 - “I am not well and not happy today” D3 - “I wish I could g


## 2. Text chunking

In this section, we split each document's text into fixed-size, overlapping chunks. Since pages within the same document are concatenated before chunking, a chunk's boundaries can span across pages rather than being restricted to a single page. The overlap between consecutive chunks helps ensure that an idea is not cut off entirely at a chunk boundary, since the last few words of one chunk are repeated at the start of the next. 

Each resulting chunk retains metadata including **its source file and the range of pages it spans (`page_start`, `page_end`), and is assigned with a unique ID.**

In [ ]:
def build_chunks(pages, chunk_size=500, overlap=50):
    """Concatenate all pages of a document into one text to allow chunks overlap span page breaks, before chunk the whole thing."""

    docs = {}
    for page in pages:
        docs.setdefault(page["source"], []).append(page)

    all_chunks = []
    for source, doc_pages in docs.items():
        doc_pages.sort(key=lambda p: p["page"])
        words_with_pages = []  # (word, page_number)
        for page in doc_pages:
            for w in page["text"].split():
                words_with_pages.append((w, page["page"]))

        start = 0
        while start < len(words_with_pages):
            end = start + chunk_size
            piece = words_with_pages[start:end]
            chunk_words = [w for w, _ in piece]
            page_numbers = [p for _, p in piece]
            all_chunks.append({
                "id": str(uuid.uuid4()),
                "text": " ".join(chunk_words),
                "source": source,
                "page_start": min(page_numbers),
                "page_end": max(page_numbers),
            })
            start += chunk_size - overlap
    return all_chunks

def format_page_range(chunk):
    """Page formatting helper: 'page 4' if the chunk from one page, 'pages 3-4' if it spans several."""

    if chunk["page_start"] == chunk["page_end"]:
        return f"page {chunk['page_start']}"
    return f"pages {chunk['page_start']}-{chunk['page_end']}"

chunks = build_chunks(all_pages, chunk_size=500, overlap=50)

print(f"Created {len(chunks)} chunks from {len(all_pages)} pages.")

# Preview
print("\n\nExample chunk")
print("="*50)
print(textwrap.fill(chunks[0]["text"][:400], width=100))
print(f"\nSource: {chunks[0]['source']}, {format_page_range(chunks[0])}")


Created 61 chunks from 72 pages.


Example chunk
Text representation A. Bag-of-words (BOW) and One-hot encoding Bag-of-Words (BoW): A text
representation technique where a document is represented as a vector of word counts or occurrences
in a vocabulary. • One-Hot Encoding: A technique that converts categorical variables (including
words) into binary vectors, where only one position in the vector is "hot" (1), and the rest are
"cold" (0). • Exam

Source: personal_notes.pdf, pages 1-4


## 3. Text encoding

Here, we transform each text chunk into a semantic vector using an embedding model. In this project, we use **Sentence-BERT** (`all-MiniLM-L6-v2`), which converts text into a fixed-length vector that captures its meaning.

The resulting vectors are then stored in a vector database using **ChromaDB**, along with each chunk's metadata. We use a persistent local ChromaDB collection so the generated vector database is saved to disk to avoid the need to re-embed and re-index everything on every run. Using a vector database lets us perform similarity search over meaning rather than exact keyword matching, retrieving chunks whose semantic content is closest to a given query. This is the core retrieval mechanism that RAG depends on.

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Persistent local Chroma store so that data survives between notebook runs
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="study_assistant")

def index_chunks(chunks, batch_size=64):
    """Indexes chunks into the ChromaDB collection."""
    
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        texts = [c["text"] for c in batch]
        embeddings = embedder.encode(texts, show_progress_bar=False).tolist()
        collection.add(
            ids=[c["id"] for c in batch],
            embeddings=embeddings,
            documents=texts,
            metadatas=[{"source": c["source"], "page_start": c["page_start"], "page_end": c["page_end"]} for c in batch],
        )

index_chunks(chunks)
print(f"Indexed {collection.count()} chunks into ChromaDB.")


Indexed 61 chunks into ChromaDB.


## 4. Retriever

Now that the source text has been extracted, encoded, and stored in a vector database, we define a retriever that pulls the most relevant chunks for a given query. The process begins by encoding the query using the same embedding model used for the chunks, then retrieving **the top-k most similar chunks** from Chroma based on vector similarity.

Below, we test the retriever with a sample query to confirm that only relevant chunks are being returned, before connecting it to the LLM in later steps.

In [ ]:
def retrieve(query, k=3):
    """Retrieves the top-k most relevant chunks from the ChromaDB collection for a given query."""

    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    retrieved = []
    for text, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        retrieved.append({
            "text": text,
            "source": meta["source"],
            "page_start": meta["page_start"],
            "page_end": meta["page_end"],
            "distance": dist,
        })
    return retrieved

def format_page_range(chunk):
    """Display helper: 'page 4' if the chunk stays on one page, 'pages 3-4' if it spans several."""
    
    if chunk["page_start"] == chunk["page_end"]:
        return f"page {chunk['page_start']}"
    return f"pages {chunk['page_start']}-{chunk['page_end']}"

# Add a sample query
test_query = "What is lemmatization?"
hits = retrieve(test_query, k=3)

for i, h in enumerate(hits, 1):
    print(f"[{i}] {h['source']} ({format_page_range(h)}, distance {h['distance']:.3f})")
    print(textwrap.fill(h["text"][:250], width=100))
    print()


[1] textbook.pdf (pages 20-21, distance 0.991)
have the same root, despite their surface differences. The words am, are, and is have the shared
lemma be; the words dinner and dinners both have the lemma dinner. Lemmatizing each of these forms
to the same lemma will let us ﬁnd all mentions of word

[2] personal_notes.pdf (pages 6-9, distance 1.203)
forms of a word to a common base form. Lemmatization is the task of determining that two words have
the same root, despite their surface differences. For example, the words sang, sung, and sings are
forms of the verb sing. The word sing is the common

[3] textbook.pdf (pages 1-2, distance 1.369)
from each other tokenization by whitespace, but whitespace is not always sufﬁcient. New York and
rock ’n’ roll are sometimes treated as large words despite the fact that they contain spaces, while
sometimes we’ll need to separate I’m into the two wor



## 5. Building prompt

A prompt needs to be constructed to instruct the LLM on how to generate its answer based on our requirements. Here, we combine the retrieved source chunks and the user's query into a single prompt, with explicit instructions emphasizing that the model should answer ***only*** using the retrieved context, and should clearly state if it doesn't know the answer when the necessary information isn't present in the source texts.

In [ ]:
def build_prompt(query, retrieved_chunks):
    """Builds a prompt for the LLM using the retrieved chunks and the user's query."""
    
    context_block = "\n\n".join(
        f"[{i+1}] (Source: {c['source']}, {format_page_range(c)})\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )
    prompt = f"""You are a study assistant. Answer the question using ONLY the context below.
If the context does not contain enough information to answer, say so clearly instead of guessing.

When citing information, use ONLY the bracket number, like [1] or [2], directly after the relevant
sentence. Do NOT repeat the source filename or page numbers in your answer text. A source list
will be shown separately after your answer.

Write your answer as clear, direct prose. Do not describe what each numbered source says one by
one and synthesize the information into a single coherent answer.

Context:
{context_block}

Question: {query}

Answer:"""

    return prompt

print(build_prompt(test_query, hits))


You are a study assistant. Answer the question using ONLY the context below.
If the context does not contain enough information to answer, say so clearly instead of guessing.

When citing information, use ONLY the bracket number, like [1] or [2], directly after the relevant
sentence. Do NOT repeat the source filename or page numbers in your answer text. A source list
will be shown separately after your answer.

Write your answer as clear, direct prose. Do not describe what each numbered source says one by
one and synthesize the information into a single coherent answer.

Context:
[1] (Source: textbook.pdf, pages 20-21)
have the same root, despite their surface differences. The words am, are, and is have the shared lemma be; the words dinner and dinners both have the lemma dinner. Lemmatizing each of these forms to the same lemma will let us ﬁnd all mentions of words in Polish like Warsaw. The lemmatized form of a sentence like He is reading detective stories would thus be He be read de

## 6. Generation

We send the prompt to **a local model via Ollama, using `llama3.1:8b`**, since it runs entirely offline at no cost and requires no API key for a locally-run model. 

Before running this, make sure the Ollama app is installed, then run the following in a terminal to start the server and download the model:

```
ollama serve
ollama pull llama3.1:8b
```

In [ ]:
OLLAMA_MODEL = "llama3.1:8b"

def generate_answer(prompt, model=OLLAMA_MODEL):
    """Generates an answer from the LLM using the provided prompt."""
    
    response = ollama.generate(model=model, prompt=prompt)
    return response["response"].strip()

answer = generate_answer(build_prompt(test_query, hits))
answer

'Lemmatisaton is the task of determining that two words have the same root, despite their surface differences [1]. It involves identifying the common base form of a word by stripping off grammatical inflections and suffixes [2]. For example, the words sang, sung, and sings are forms of the verb sing [1, 3]. Lemmatization is essential for processing morphologically complex languages like Arabic [1, 3].'

## 7. Full pipeline

Here, we wrap retrieval, prompting, and generation into a single function for reusability, so that a question can be answered by simply passing in a query. The function handles the full pipeline internally and returns the generated answer alongside the sources it was built from.

In [ ]:
def ask(query, k=3, model=OLLAMA_MODEL, verbose=True):
    """Pipeline function that retrieves relevant chunks for a query, builds a prompt, and generates an answer."""
    
    retrieved = retrieve(query, k=k)
    prompt = build_prompt(query, retrieved)
    answer = generate_answer(prompt, model=model)

    if verbose:
        print("QUESTION:", query)
        print("\nANSWER:\n" + textwrap.fill(answer, width=100))
        print("\nSOURCES:")
        for i, c in enumerate(retrieved, 1):
            print(f"  [{i}] {c['source']}, {format_page_range(c)}")
    return {"query": query, "answer": answer, "sources": retrieved}

_ = ask("What is lemmatization?")

QUESTION: What is lemmatization?

ANSWER:
Lemmatization is the task of determining that two words have the same root, despite their surface
differences, and mapping them to a common base form [1]. It is essential for processing
morphologically complex languages like Arabic, and can be considered a simpler version of itself,
stemming, which mainly strips suffixes from the end of the word [3].

SOURCES:
  [1] textbook.pdf, pages 20-21
  [2] personal_notes.pdf, pages 6-9
  [3] textbook.pdf, pages 1-2


## 8. Evaluation set

We perform evaluation by writing **questions whose answers we have confirmed are present in the source texts.** 

This lets us assess two things separately: retrieval accuracy, whether the correct chunk (the one that actually contains the answer) is being retrieved, and answer accuracy, whether the LLM's generated answer is factually correct given that context.

This evaluation set serves as evidence that the RAG pipeline works as intended, and provides a basis for making targeted adjustments, such as tuning chunk size, retrieval k, or prompt wording, to improve performance.

In [ ]:
# Define evaluation queries that have been confirmed to have relevant information in the indexed documents
eval_set = [
    {
        "question": "What is one-hot vector?",
        "expected_source_contains": "vector",
    },
    {
        "question": "What is sentence segmentation and how to apply it?",
        "expected_source_contains": "punctuation",
    },
    {
        "question": "What are the tasks performed in text normalization?",
        "expected_source_contains": "tokenizing",
    },
    {
        "question": "How does feedforward work in word prediction?",
        "expected_source_contains": "probability",
    },
]

eval_results = []
for item in eval_set:
    result = ask(item["question"], verbose=False)
    retrieved_text = " ".join(c["text"].lower() for c in result["sources"])
    retrieval_hit = item["expected_source_contains"].lower() in retrieved_text

    sources_str = ";\n".join(
        f"[{i}] {c['source']}, {format_page_range(c)}"
        for i, c in enumerate(result["sources"], 1)
    )

    eval_results.append({
        "Question": item["question"],
        "Retrieval hit": True if retrieval_hit else False,
        "Answer": result["answer"],
        "Sources": sources_str,
    })

df = pd.DataFrame(eval_results)
pd.set_option("display.max_colwidth", None)

styled = df.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})
display(styled)

accuracy = sum(1 for r in eval_results if r["Retrieval hit"] == True) / len(eval_results)
print(f"\nRetrieval hit rate: {accuracy:.0%}")

,Question,Retrieval hit,Answer,Sources
0,What is one-hot vector?,True,"A one-hot vector is a vector that has one element equal to 1—in the dimension corresponding to that word’s index in the vocabulary— while all the other elements are set to zero [1, 2].","[1] personal_notes.pdf, pages 1-4; [2] textbook.pdf, pages 40-41; [3] textbook.pdf, pages 41-43"
1,What is sentence segmentation and how to apply it?,True,"Sentence segmentation is an important step in text processing, which involves dividing a text into individual sentences. The most useful cues for segmenting a text into sentences are punctuation, such as periods, question marks, and exclamation points [1]. However, periods can be ambiguous, as they can mark both sentence boundaries and abbreviations. To address this, sentence tokenization and word tokenization can be done jointly, and an abbreviation dictionary can be used to determine whether a period is part of a commonly used abbreviation [1]. In the Stanford CoreNLP toolkit, sentence splitting is rule-based, and a sentence ends when a sentence-ending punctuation is not already grouped with other characters into a token [1].","[1] textbook.pdf, pages 21-22; [2] textbook.pdf, pages 17-18; [3] textbook.pdf, page 11"
2,What are the tasks performed in text normalization?,True,"Text normalization involves at least three tasks: tokenizing (segmenting) words, normalizing word formats, and segmenting sentences [1].","[1] textbook.pdf, pages 13-15; [2] textbook.pdf, pages 1-2; [3] textbook.pdf, pages 17-18"
3,How does feedforward work in word prediction?,True,"In a feedforward neural language model, the input layer represents the prior context of words, typically by one-hot vectors of length |V|, where |V| is the vocabulary size. The embedding matrix E has a column for each word, and multiplying each one-hot vector by E selects the corresponding word embedding. The resulting embedding vectors are concatenated to produce the embedding layer. This is followed by a hidden layer and an output layer, where the softmax function produces a probability distribution over possible next words.","[1] textbook.pdf, pages 37-39; [2] textbook.pdf, pages 40-41; [3] textbook.pdf, pages 40-41"



Retrieval hit rate: 100%


We can see that the results provided answers that align with the expected keywords found in the source text. They also follow the formatting we requested, using only inline citations.

Next, we want to see how the model performs on **queries that we know have no answer in the source text.**

In [ ]:
# Add questions that has no answer from source text
unanswerable_eval_set = [
    "What is overfitting?",
    "What year was the transformer architecture published?",
    "How does quantum computing relate to neural networks?",
    "What is the capital of France?",
]

unanswerable_results = []
for question in unanswerable_eval_set:
    result = ask(question, verbose=False)
    answer_lower = result["answer"].lower()

    unanswerable_results.append({
        "Question": question,
        "Answer": result["answer"],
    })

df_unanswerable = pd.DataFrame(unanswerable_results)
pd.set_option("display.max_colwidth", None)
styled_unanswerable = df_unanswerable.style.set_properties(
    **{"white-space": "pre-wrap", "text-align": "left"}
)
display(styled_unanswerable)

,Question,Answer
0,What is overfitting?,There is no information provided in the given context about overfitting.
1,What year was the transformer architecture published?,There is no information in the provided context about the publication year of the transformer architecture.
2,How does quantum computing relate to neural networks?,There is no information in the provided context about the relationship between quantum computing and neural networks.
3,What is the capital of France?,I don't have enough information to answer the question.


This confirms that the prompt and model behave appropriately, correctly declining to answer when no relevant information is available. Together with the earlier evaluation, this shows the pipeline handles both answerable and unanswerable questions correctly.